In [ ]:
import pandas as pd
import re

from pyparsing import col

In [ ]:
raw_df = pd.read_csv('stops.csv')

In [ ]:
raw_df[raw_df["stop_name"].str.contains(r'^(?:Downsview Park Station)', case=False, na=False)]

raw_df.shape



In [ ]:
df = raw_df.copy()

In [ ]:
df.head

In [ ]:
df.shape

In [ ]:
# df = df[
#     (
#             df['stop_name'].str.contains(r'\b(?:terminal|station)\b', case=False, na=False) |
#             df['stop_name'].str.contains(r'\bYork University\b', case=False, na=False)
#     )
#     &
#     ~df['stop_name'].str.contains(r'\b(?:at|go)\b', case=False, na=False)
#     ]

In [ ]:
df[df["stop_name"].str.contains(r'\b(?:North York Centre Station)\b', case=False, na=False)]

In [ ]:
df[df["stop_name"].str.contains(r'\b(?:York University)\b', case=False, na=False)]

In [ ]:
df.shape

In [ ]:
len(df['stop_name'].unique())

In [ ]:
df.stop_name

In [ ]:
len(df.stop_name.unique())

In [ ]:

ttc_stations = {
    1: ["Finch Station", "North York Centre Station", "Sheppard-Yonge Station", "York Mills Station",
        "Lawrence Station", "Eglinton Station", "Davisville Station", "St Clair Station", "Summerhill Station",
        "Rosedale Station", "Bloor Station", "Wellesley Station", "College Station", "TMU Station", "Queen Station",
        "King Station", "Union Station", "St Andrew Station", "Osgoode Station", "St Patrick Station",
        "Queen's Park Station", "Museum Station", "St George Station", "Spadina Station", "Dupont Station",
        "St Clair West Station", "Cedarvale Station", "Glencairn Station", "Lawrence West Station", "Yorkdale Station",
        "Wilson Station", "Sheppard West Station", "Downsview Park Station", "Finch West Station", "York University",
        "Pioneer Village Station", "Highway 407 Station", "Vaughan Metropolitan Centre Station"],
    2: ["Kipling Station", "Islington Station", "Royal York Station", "Old Mill Station", "Jane Station",
        "Runnymede Station", "High Park Station", "Keele Station", "Dundas West Station", "Lansdowne Station",
        "Dufferin Station", "Ossington Station", "Christie Station", "Bathurst Station", "Spadina Station",
        "St George Station", "Bay Station", "Yonge Station", "Sherbourne Station", "Castle Frank Station",
        "Broadview Station", "Chester Station", "Pape Station", "Donlands Station", "Greenwood Station",
        "Coxwell Station", "Woodbine Station", "Main Street Station", "Victoria Park Station", "Warden Station",
        "Kennedy Station"],
    4: ["Sheppard-Yonge Station", "Bayview Station", "Bessarion Station", "Leslie Station", "Don Mills Station"],
    5: ["Mount Dennis Station", "Keelesdale Station", "Caledonia Station", "Fairbank Station", "Oakwood Station",
        "Cedarvale Station", "Forest Hill Station", "Chaplin Station", "Avenue Station", "Eglinton Station",
        "Mount Pleasant Station", "Leaside Station", "Laird Station", "Sunnybrook Park Station", "Don Valley Station",
        "Aga Khan Park & Museum Station", "Wynford Station", "Sloane Station", "O'Connor Station", "Pharmacy Station",
        "Hakimi Lebovic Station", "Golden Mile Station", "Birchmount Station", "Ionview Station", "Kennedy Station"],
    6: ["Humber College Station", "Westmore Station", "Martin Grove Station", "Albion Station", "Stevenson Station",
        "Mount Olive Station", "Rowntree Mills Station", "Pearldale Station", "Duncanwoods Station",
        "Milvan Rumike Station", "Emery Station", "Signet Arrow Station", "Norfinch Oakdale Station",
        "Jane and Finch Station", "Driftwood Station", "Tobermory Station", "Sentinel Station", "Finch West Station"]
}

ttc_station_list = [station for sublist in ttc_stations.values() for station in sublist]
len(ttc_stations)

In [ ]:
def is_subway_station(stop_name: str):
    res = re.findall(r'\b(?:platform| York University | Union Station)\b', stop_name, flags=re.I)
    return any(station in stop_name for station in ttc_station_list) and bool(res)


In [ ]:
# is_subway_station("Sheppard-Yonge Station - Southbound Platform")
# is_subway_station("Kipling Station - SUBWAY Platform")

In [ ]:
# def is_subway_bus_station(stop_name:str):
#     res = re.findall(r'\b(?:platform)\b', stop_name, flags=re.I)
#     return any(station in stop_name for station in ttc_station_list) and not bool(res)

In [ ]:
# is_subway_bus_station("Sheppard Ave East at Burbank Dr - Bessarion Station")

In [ ]:

sorted_stations = sorted(ttc_station_list, key=len, reverse=True)


def find_normalized_station(name):
    if not isinstance(name, str):
        return None
    name_lower = name.lower()
    for station in sorted_stations:
        if station.lower() in name_lower:
            return station
    return None

In [ ]:
def get_line(series):
    is_subway = series['is_subway']
    if not is_subway:
        return None
    root_station = series['parent_station']
    keys = [str(k) for k, v in ttc_stations.items() if any(root_station.lower() in station.lower() for station in v)]
    return ",".join(keys) if keys else None

In [ ]:
import re

'''
Assumption is that the station name is always present in the stop_name and it contains the word "station" (except for the special case of "York University").
And station names ends with Platform

Example
Kennedy Station - SouthBound Platform
Sheppard-Yonge Station - SouthBound Platform
Don Mills Station - SUBWAY Platform - Terminal
'''


def clean_station_name(name):
    if not isinstance(name, str):
        return None
    #  this is the only exception is station name pattern present in the dataset
    if "York University" in name:
        return "York University"

    name = re.sub(r'\b(Eastbound|Westbound|Northbound|Southbound)\b', '', name, flags=re.I)
    name = re.sub(r'\b(LRT|Bus|Platform|Stop|Terminal|SUBWAY Platform)\b', '', name, flags=re.I)
    name = re.sub(r'\s+[^\w&]+\s+', ' ', name)

    # Normalize spacing
    name = re.sub(r'\s+', ' ', name).strip()

    return name


def get_platform_direction(name):
    if not isinstance(name, str):
        return None
    match = re.search(r'\b(Eastbound|Westbound|Northbound|Southbound)\b', name, flags=re.I)
    return match.group(1).capitalize() if match else None


In [ ]:
# Apply function across rows where axis=1

def generate_parent_station_id(row):
    station_name = row['parent_station']
    if pd.isna(station_name):
        return None
    # Find all stops with the same parent_station and return the min stop_id
    stations = df[df['parent_station'] == station_name]
    return stations['stop_id'].min() if not stations.empty else None

In [ ]:
'''
Gets the station mode
Assumption is that if the stop name contains "at" it is a bus stop, if it contains "station" and is in the list of subway stations it is a subway or LRT station, if it contains "go" it is a GO station, otherwise - default it is a bus stop.
'''


def station_mode(row):
    stop_name = row['stop_name']
    is_subway = row['is_subway']
    if not is_subway:
        return 'BUS'
    line = row['line_number']
    if line is not None:
        if any(x in "1,2,4" for x in line):
            return 'SUBWAY'
        return 'LRT'
    elif "go" in stop_name.lower():
        return 'GO'
    else:
        return 'BUS'

In [ ]:
# station_mode("Rowntree Mills Station")
# station_mode("Rowntree Mills Station Eastbound Platform")

In [ ]:
'''
TODO :  ADD A UNIQUE STATION ID FOR EACH STATION BASED ON

FOR SUBWAY (TBA)
LINE_LineNumber_PARENTID_STOP_CODE
------------------------------------------------
FOR TTC (UNIQUE CODE ONLY)
BUS_STOP_CODE
------------------------------------------------
FOR GO
GO_STOP_CODE
'''


def generate_station_ids(row):
    mode = row['mode']
    if mode == 'SUBWAY' or mode == 'LRT':
        return f"{mode}_{row['parent_station_id']}_{row['stop_id']}"

    return f"{mode}_{row['stop_id']}"


In [ ]:

df['temp_station'] = df['stop_name'].apply(find_normalized_station)

df['parent_station'] = (df.groupby(['stop_lat', 'stop_lon'])['temp_station'].transform(
    lambda x: x.dropna().iloc[0] if len(x.dropna()) > 0 else None))

df['is_subway'] = df['stop_name'].apply(is_subway_station)

df['line_number'] = df.apply(get_line, axis=1)

df['temp_station_id'] = df.groupby('parent_station')['stop_id'].transform('min')

df['parent_station_id'] = (df.groupby(['stop_lat', 'stop_lon'])['temp_station_id'].transform(
    lambda x: x.dropna().iloc[0] if len(x.dropna()) > 0 else None).astype('Int64'))

df['mode'] = df.apply(station_mode, axis=1)

df['direction'] = df['stop_name'].apply(get_platform_direction)

df.drop(columns=['temp_station'], inplace=True)
df.drop(columns=['temp_station_id'], inplace=True)

df['station_id'] = df.apply(generate_station_ids, axis=1)




In [ ]:
df.head()

In [ ]:
(df[df['parent_station'].str.contains(r'\b(?:Union Station)\b', case=False, na=False)])

In [ ]:
# print(df[df['parent_station'].str.contains(r'\b(?:kipling)\b', case=False, na=False)])
# df[df["is_subway_line"] == True]['is_subway_line'].count()

In [ ]:
df.columns

In [ ]:
len(df['parent_station'].dropna().unique())

In [ ]:
df[['parent_station', 'line_number']].dropna().drop_duplicates(subset=['parent_station'])['line_number'].value_counts()

In [ ]:
df['station_id'][df['mode'] == 'SUBWAY'].count()

In [ ]:
df.columns


In [ ]:
df[df['mode'] == 'SUBWAY']["stop_name"].head()

In [ ]:
(df[df["parent_station"] == "North York Centre Station"])

In [ ]:
# df_ttc = df[df['mode'] == 'TTC'].copy()
#
# df_ttc['station'] = df_ttc['stop_name'].apply(
#     lambda x: next((s for s in ttc_stations if s in x), None)
# )

In [ ]:
processed = list(map(lambda x: str(x).lower(), df['parent_station'].dropna().unique().tolist()))

In [ ]:
not_found = [s for s in ttc_station_list if s.lower() not in processed]
not_found

In [ ]:
set(df['parent_station'].dropna().unique().tolist()) - set(ttc_station_list)

In [ ]:
df.shape

In [ ]:
# generated by Gemini

import folium
from folium.plugins import MarkerCluster

# For mapping, we wan˝t to plot the individual stops (platforms) that have a parent station
df_stations = df.dropna(subset=['parent_station'])


# Get an arbitrary line number for color mapping if it's an interchange station
def first_line_num(lines_str):
    if isinstance(lines_str, str) and lines_str:
        return int(lines_str.split(',')[0])
    return 1


# We group by parent_station, line_number, and direction to plot markers with arrows
station_coords = df_stations.groupby(['parent_station', 'line_number', 'direction'], dropna=False)[
    ['stop_lat', 'stop_lon']].mean().reset_index()

center_lat = station_coords['stop_lat'].mean()
center_lon = station_coords['stop_lon'].mean()

# Allowed Folium colors: 'red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen', 'gray', 'black', 'lightgray'
colors = {
    1: 'yellow',
    # Note: 'yellow' might not be standard Folium color, fallback used if error: we will use custom hex for lines
    2: 'green',
    4: 'purple',
    5: 'orange',
    6: 'gray'
}

marker_colors = {
    1: 'orange',  # Using supported colors
    2: 'green',
    4: 'purple',
    5: 'cadetblue',
    6: 'gray'
}

# Icon mapping based on direction
direction_icons = {
    'Northbound': 'arrow-up',
    'Southbound': 'arrow-down',
    'Eastbound': 'arrow-right',
    'Westbound': 'arrow-left'
}

m = folium.Map(location=[center_lat, center_lon], zoom_start=11)

marker_cluster = MarkerCluster().add_to(m)

for idx, row in station_coords.iterrows():
    f_line = first_line_num(row['line_number'])
    direction = row['direction'] if pd.notna(row['direction']) else "Unknown"

    icon_name = direction_icons.get(direction, 'info-sign')

    folium.Marker(
        location=[row['stop_lat'], row['stop_lon']],
        popup=f"{row['parent_station']} (Line {row['line_number']}) - {direction}",
        icon=folium.Icon(color=marker_colors.get(f_line, 'blue'), icon=icon_name,
                         prefix='fa' if 'arrow' in icon_name else 'glyphicon'),
    ).add_to(marker_cluster)

# Draw polylines per line
# In order to draw connecting lines exactly, we iterate over the predefined lines and draw them
for line_id, st_list in ttc_stations.items():
    # Filter the subset coordinates for the specific line
    # Map station name to its coordinates
    line_coords = []
    for st_name in st_list:
        st_row = station_coords[station_coords['parent_station'] == st_name]
        if not st_row.empty:
            # Pick the first one for the line connection
            line_coords.append((st_row.iloc[0]['stop_lat'], st_row.iloc[0]['stop_lon']))

    # We map colors manually for Polyline
    poly_colors = {1: '#F8C300', 2: '#00923F', 4: '#B2005A', 5: '#F2602A', 6: '#A19E92'}

    if line_coords:
        folium.PolyLine(line_coords, color=poly_colors.get(line_id, 'red'), weight=4, opacity=0.8,
                        tooltip=f"Line {line_id}").add_to(m)

m.save('index.html')
# m

In [ ]:
# validating if stop_id and stop_code are same
df['stop_id'].equals(df['stop_code'])
(df.stop_id == df.stop_code).all()
# The result is true, so we can safely drop stop_code
df.drop(columns=['stop_code', "stop_url", 'stop_desc', 'zone_id', 'location_type', 'stop_timezone'], inplace=True)

In [ ]:
df.shape

In [ ]:
df.to_csv('processed_stations.csv', index=False)
print("Saved Processed DataFrame to 'processed_stations.˝csv'")

In [ ]:
df.head()

In [ ]:
df[['stop_id', 'stop_name', 'mode', 'station_id']].isna().any()

In [ ]:
df[df["stop_name"].str.contains(r'\b(?:Downsview Park Station)\b', case=False, na=False)].drop(
    columns=["wheelchair_boarding", "stop_lat", "stop_lon"])



In [ ]:
is_subway_station("Downsview Park Station - Northbound Platform")
is_subway_station("Sheppard Ave West at Bakersfield St West Side - Downsview Park Station")


In [ ]:
df.head()
df[['stop_id', 'stop_name', 'mode', 'station_id']].isna().any()

In [ ]:
df['parent_station_id'].dropna().drop_duplicates().count()

In [ ]:
subways_stations = df[(df['mode'] == 'SUBWAY') | (df['mode'] == 'LRT')][
    ['parent_station_id', 'parent_station', 'mode','line_number']].dropna().drop_duplicates(
    subset=['parent_station_id']).sort_values(by='parent_station_id')
subways_stations.rename(columns={'parent_station_id': 'id', 'parent_station': 'name'}, inplace=True)
subways_stations.to_csv("subways_stations.csv", index=False)

In [ ]:
df[df['parent_station_id'] == 16222]

In [ ]:
(df[df['stop_name'].str.contains(r'\b(?:Union Station)\b', case=False, na=False)])

In [ ]:
df[(df['stop_id'] >= 13733) & (df['stop_id'] <= 13738)]

In [ ]:
df[df['stop_id'] == 11376]

In [ ]:
ss_n = df[(df['mode'] == 'SUBWAY') | (df['mode'] == 'LRT')][
    ['parent_station_id', 'parent_station', 'mode', 'stop_lat', 'stop_lon', 'line_number', 'direction', 'stop_id']].sort_values(
    by='parent_station_id').dropna().drop_duplicates(
    subset=['parent_station_id']).sort_values(by='parent_station_id')
ss_n.rename(columns={'stop_id': 'id', 'parent_station': 'name'}, inplace=True)

In [ ]:
min_lat = ss_n['stop_lat'].min()
max_lat = ss_n['stop_lat'].max()
min_lon = ss_n['stop_lon'].min()
max_lon = ss_n['stop_lon'].max()

def translate_coordinates(lat, lon, canvas_width: int = 1600, canvas_height=1800):
    def x(lon):
        return int((lon - min_lon) / (max_lon - min_lon) * canvas_width)

    def y(lat):
        return int((1 - (max_lat - lat) / (max_lat - min_lat)) * canvas_height)

    return x(lon), y(lat)




In [ ]:
ss_n['translated_coords'] = ss_n[['stop_lat', 'stop_lon']].apply(lambda row: translate_coordinates(row['stop_lat'], row['stop_lon']), axis=1)

In [ ]:
ss_n['translated_lat'] = ss_n['translated_coords'].apply(lambda x: x[1])
ss_n['translated_lon'] = ss_n['translated_coords'].apply(lambda x: x[0])
ss_n.drop(columns=['translated_coords'], inplace=True)

In [ ]:
ss_n.to_csv("ss_n.csv", index=False)

In [ ]:
ss_n.to_json("ss_n.json", orient='records', lines=True)

In [ ]:
ss_n['line_number'].str.contains(r'2').sum()

In [ ]:
import this

In [ ]:
subways_stations[subways_stations['line_number'].str.contains(r'1', case=False, na=False)].to_csv("line_1_stations.csv", index=False)

In [1]:
subways_stations['line_number'].unique()

NameError: name 'subways_stations' is not defined